# Mamba vs. Transformer Porównanie zdolności klasyfikacji na zbiorze danych IMDb

## 1. Setup i biblioteki

Możemy to pominąć i wykonać komendę `uv sync` jeśli notebook uruchamiamy lokalnie

### 1.1 Instalacja bibliotek

In [8]:
# Example for CUDA 12.3+ and PyTorch 2.x (Adjust URL based on your exact versions if needed)
!pip install https://github.com/state-spaces/mamba/releases/download/v2.2.2/mamba_ssm-2.2.2+cu122torch2.3cxx11abiFALSE-cp312-cp312-linux_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.8/323.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 4.4 MB/s eta 0:00:00


In [ ]:
!pip uninstall -y torch torchvision torchaudio # Uninstall any existing versions to avoid conflicts
!pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121


Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 26.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 122.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 83.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 60.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 114.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

KeyboardInterrupt: 

In [1]:
!pip install transformers datasets accelerate evaluate

### 1.2 Importy

In [1]:
import torch
import datasets
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate
import numpy as np

In [2]:
import evaluate
import numpy as np

# Helper function to compute metrics
def compute_metrics(eval_pred):
    load_accuracy = evaluate.load("accuracy")
    load_f1 = evaluate.load("f1")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = load_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = load_f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": accuracy, "f1": f1}

## 2. Wczytanie datasetu i preprocessing

We will load the `imdb` dataset from the Hugging Face `datasets` library, which contains movie reviews labeled as positive or negative sentiment. After loading, we will tokenize the text and prepare the data for model training.

In [3]:
print("Is CUDA available?", torch.cuda.is_available())

Is CUDA available? True


In [4]:
dataset = datasets.load_dataset('stanfordnlp/imdb')

print("Dataset loaded successfully:")
print(dataset)

Dataset loaded successfully:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [5]:
print("\nExample from training set:")
print(dataset['train'][0])


Example from training set:
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudi

### 2.2 Tokenizacja

We need to convert the text reviews into numerical tokens that the models can understand. We'll use a `AutoTokenizer` suitable for our chosen Transformer baseline (e.g., DistilBERT) and apply it to the entire dataset. For Mamba, a similar tokenization strategy will be used.

In [6]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
# TODO parametrize this with max_length
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512) # Default max_length, will be varied in scaling analysis

# Apply tokenization to the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

print("\nTokenized dataset example:")
print(tokenized_dataset['train'][0])


Tokenized dataset example:
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudi

In [7]:
# Prepare data for training
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")

# Create data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("\nDataset ready for training:")
print(tokenized_dataset['train'].column_names)


Dataset ready for training:
['labels', 'input_ids', 'token_type_ids', 'attention_mask']


## 3. Podejście 1: Transformer (Baseline)

In this section, we will implement and train a Transformer model. We'll start with fine-tuning a pre-trained model like DistilBERT, which is a good balance between performance and computational cost. We'll define the model, training arguments, and use the `Trainer` API for training.

### Fine-tuning DistilBERT

We will fine-tune a `distilbert-base-uncased` model for sequence classification. This involves loading the pre-trained model, configuring training arguments, and using the `Trainer` API to manage the training and evaluation process.

In [11]:
# Define a custom callback for timing epochs
import time

from transformers import TrainerCallback


class TimingCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, **kwargs):
        self.epoch_start_time = time.time()

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch_end_time = time.time()
        epoch_duration = epoch_end_time - self.epoch_start_time
        print(f"Epoch {state.epoch:.0f} completed in {epoch_duration:.2f} seconds")

# Load pre-trained DistilBERT model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased", num_labels=2)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=1, # Changed from 10 to 1 for more frequent updates
    report_to="tensorboard" # Add this for loss tracking
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[TimingCallback()] # Add TensorBoardCallback and TimingCallback
)

# Train the model
trainer.train()

# Evaluate the model
results = trainer.evaluate()
print("\nDistilBERT Evaluation Results:")
print(results)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.047184,0.237508,0.908680,0.908383
2,0.011416,0.233606,0.931800,0.931799
3,0.002213,0.282658,0.931640,0.931639


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 3 completed in 314.64 seconds


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.002213,0.282658,3,0.931640,0.931639



DistilBERT Evaluation Results:
{'eval_loss': 0.28265848755836487, 'eval_accuracy': 0.93164, 'eval_f1': 0.9316394792568972}


## 4. Approach 2: Mamba

Here, we will implement and train a Mamba model. We can either fine-tune a pre-trained Mamba model (e.g., `state-spaces/mamba-130m` if available for classification) or build a small Mamba model from scratch and train it for sequence classification. This section will require custom model definition and potentially custom training loops if the `Trainer` API isn't directly compatible.

### Fine-tuning Mamba

For Mamba, we will define a custom `MambaForSequenceClassification` model, as `transformers` does not yet have a direct `AutoModelForSequenceClassification` for Mamba. We will initialize a Mamba block and add a classification head on top. The training process will then use the same `Trainer` API.

In [ ]:
from transformers import MambaForSequenceClassification, AutoConfig
import torch.nn as nn
import time
import torch

# Initialize Mamba model for sequence classification
# NOTE: `state-spaces/mamba-130m` was likely pre-trained with a tokenizer like `EleutherAI/gpt-neox-20b`.
# Using `distilbert-base-uncased` tokenizer might lead to sub-optimal results due to vocabulary differences.
# The `transformers` library will attempt to resize the embeddings, but the new embeddings will be random.

mamba_model = MambaForSequenceClassification.from_pretrained(
    "state-spaces/mamba-130m",
    num_labels=2, # For binary classification (positive/negative sentiment)
    # If the tokenizer's vocab_size differs from the pre-trained model's, transformers will resize.
    # We don't explicitly pass vocab_size here as from_pretrained handles it by loading the model's config.
)

# Define training arguments (can be same as Transformer or adjusted)
mamba_training_args = TrainingArguments(
    output_dir="./mamba_results",
    eval_strategy="epoch", # Corrected argument name
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./mamba_logs',
    logging_steps=10,
)

# Initialize Trainer for Mamba
mamba_trainer = Trainer(
    model=mamba_model,
    args=mamba_training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    # tokenizer=tokenizer, # Removed redundant tokenizer argument
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train the Mamba model and track time
print("\nStarting Mamba model training...")
start_train_time_mamba = time.time()
mamba_train_output = mamba_trainer.train()
end_train_time_mamba = time.time()
training_time_mamba = end_train_time_mamba - start_train_time_mamba

# Evaluate the Mamba model and track time
print("\nStarting Mamba model evaluation...")
start_eval_time_mamba = time.time()
mamba_results = mamba_trainer.evaluate()
end_eval_time_mamba = time.time()
inference_time_mamba = end_eval_time_mamba - start_eval_time_mamba

print("\nMamba Training Statistics:")
print(f"Epoch Training Time: {training_time_mamba / mamba_training_args.num_train_epochs:.2f} seconds per epoch")
print(f"Total Training Time: {training_time_mamba:.2f} seconds")

print("\nMamba Evaluation Results:")
print(f"Inference Time: {inference_time_mamba:.2f} seconds")
print(f"Accuracy: {mamba_results['eval_accuracy']:.4f}")
print(f"F1 Score: {mamba_results['eval_f1']:.4f}")
print(mamba_results)

## 5. Approach 3: Scaling Analysis

This section will focus on comparing both architectures across different sequence lengths (128, 512, 1024 tokens). We will measure:
-   **Training time per epoch**
-   **Inference time**
-   **Quality metrics**: Accuracy, F1-score

This will involve re-tokenizing the dataset with different `max_length` values and repeating the training and evaluation steps for each model and sequence length.

In [ ]:
# Helper function to compute metrics
def compute_metrics(eval_pred):
    load_accuracy = evaluate.load("accuracy")
    load_f1 = evaluate.load("f1")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = load_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = load_f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": accuracy, "f1": f1}